<a href="https://colab.research.google.com/github/SleepyEveryD/NLP/blob/mathonly/notebooks/04_adaptive_routing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 · Adaptive prompt routing — a small-LLM reasoning study

**Research question:** for a small model (Qwen2.5-7B-Instruct), does routing each question to a prompt
chosen for its *reasoning shape* beat forcing one universal prompt on everything? And the deeper one:
**when does explicit reasoning help, and when does it hurt?**

The motivating failure is real and on the leaderboard: the Maths run died at level 6 on a clock-chime
*interval-counting* question — `cot_v2`'s hard "≤3 steps" brevity cap made the model **guess before it
had counted**. That same cap is what keeps verbose stats questions from timing out. One prompt cannot
serve both regimes — which is exactly the hypothesis this experiment tests.

We isolate ONE variable — the **prompt strategy** — and compare four conditions over a labelled
8-category reasoning set:

| condition | prompt |
|---|---|
| **A_universal** | one universal prompt (production `few_shot_v1`) |
| **B_generic_cot** | always plain "think step by step" |
| **C_structured** | always structured enumeration |
| **D_adaptive** | `ReasoningRouter` picks per question |

No retrieval, no calculator here — *only* the prompt changes between conditions, so the effect we
measure is the prompt's alone.

## 1 · Setup — clone/pull the repo, put `src` on the path
Runs on Colab (clones) **or** locally (uses the repo you are already in).

In [ ]:
import os, sys

# Stand in a directory that definitely EXISTS before anything else. A prior `!rm -rf /content/NLP`
# can delete the notebook's current working directory, after which every shell command fails with
# "Unable to read current working directory". Recover from that here.
SAFE = '/content' if os.path.isdir('/content') else os.path.expanduser('~')
try:
    os.getcwd()
except OSError:
    os.chdir(SAFE)

# Colab: clone/refresh the *mathonly* branch into /content/NLP (the experiment lives there, NOT on main).
# Local: use the repo this notebook already sits in.
REPO_URL = 'https://github.com/SleepyEveryD/NLP.git'
BRANCH = 'mathonly'
if os.path.isdir('/content'):
    os.chdir('/content')                       # clone from OUTSIDE the repo dir, never from within it.
    REPO_ROOT = '/content/NLP'
    if not os.path.exists(REPO_ROOT):
        !git clone -b {BRANCH} {REPO_URL} {REPO_ROOT}
    else:
        # Existing checkout: ensure we are on mathonly and current (a prior run may have cloned main).
        !cd {REPO_ROOT} && git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
else:
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()

SRC = os.path.join(REPO_ROOT, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.chdir(REPO_ROOT)
print('Repo root:', REPO_ROOT)
print('On sys.path:', SRC)
assert os.path.exists('data/reasoning_eval.jsonl'), 'Missing data/reasoning_eval.jsonl -- on the mathonly branch, are you?'


In [ ]:
# Install the inference stack so the REAL Qwen 4-bit load works.
# `-U` + bitsandbytes>=0.46.1 is MANDATORY: Colab preinstalls a stale bitsandbytes (<0.46.1) and a bare
# pin leaves it be, so the 4-bit loader ImportErrors and the experiment silently falls back to the
# SIMULATED fixture (not the model). pandas/requests pinned to Colab's own versions -- bare names break
# google-colab / cudf / gradio. Skip this cell ONLY if you intend to run the fixture.
!pip install -q -U 'transformers>=4.45.0' 'accelerate>=0.34.0' 'bitsandbytes>=0.46.1' sentencepiece einops pyyaml 'pandas==2.2.2' matplotlib 'requests==2.32.4'
print('Dependencies installed. IMPORTANT: now Runtime > Restart session, then Run all again --')
print('bitsandbytes must be re-imported fresh, or the 4-bit loader keeps the old version.')


## 2 · Load the labelled reasoning set
40 MCQs across 8 categories, each with a hand-verified gold answer **and** a gold reasoning-category label (the truth the router is scored against). The clock-chime question that broke the live Maths run is `ic-001`.

In [ ]:
from experiments.adaptive_routing import load_reasoning_eval

questions, categories = load_reasoning_eval('data/reasoning_eval.jsonl')
print(f'{len(questions)} questions; categories:')
from collections import Counter
for cat, n in sorted(Counter(categories.values()).items()):
    print(f'  {cat:<22} {n}')
print('\nThe motivating failure, ic-001:')
print(' ', next(q.text for q in questions if q.qid == 'ic-001')[:140], '...')


## 3 · Pick the engine

**Real run (Colab GPU):** `TransformersEngine` loads Qwen2.5-7B 4-bit — the numbers are real.

**Local / no-GPU:** `SimulatedReasoningEngine` — a **deterministic fixture, NOT a model**. Its
correctness comes from a hand-set skill table that *encodes the hypothesis*, so it validates the
**harness** (routing, logging, metrics, figures) end-to-end. It does **not** produce real findings.

In [ ]:
USE_REAL_MODEL = True   # set False to force the simulated fixture

engine = None
if USE_REAL_MODEL:
    try:
        from inference.engine import TransformersEngine
        from config import RunConfig
        cfg = RunConfig.from_yaml('configs/live.yaml')
        engine = TransformersEngine(cfg.model.name, cfg.model.quantization, cfg.model.dtype)
        engine.warmup()
        print('REAL engine loaded:', engine.name)
    except Exception as e:
        print(f'Real engine unavailable ({type(e).__name__}: {e})')
        print('-> falling back to the SIMULATED fixture.')
        engine = None

if engine is None:
    from inference.engine import SimulatedReasoningEngine
    engine = SimulatedReasoningEngine(questions, categories)
    print('\n' + '!' * 72)
    print('!! SIMULATED FIXTURE ENGINE -- validates the HARNESS, not the model.   !!')
    print('!! Real findings need USE_REAL_MODEL=True on a Colab GPU runtime.      !!')
    print('!' * 72)


## 4 · Run the experiment — 4 conditions × 40 questions
Writes one `experiments/adaptive_routing/records.jsonl` (strategy chosen, router verdict, correctness, latency, tokens, reasoning length, raw output).

In [ ]:
from experiments.adaptive_routing import AdaptiveRoutingExperiment

# max_new_tokens=512 (vs the 256 default): the game allows 130s/question, so a longer cap is free and
# fixes the no_answer_parsed truncations where verbose chains never reached the 'Answer:' line.
exp = AdaptiveRoutingExperiment(engine, max_new_tokens=512)
records_path = exp.run(questions, categories, verbose=True)
records_path


## 5 · Headline — does adaptive (D) beat the rest, and at what cost?

In [ ]:
import pandas as pd
from experiments import analysis
pd.set_option('display.width', 170); pd.set_option('display.max_columns', 20)

df = analysis.load_records(records_path)
comp = analysis.strategy_comparison_table(df)
comp.round(3)


In [ ]:
from IPython.display import Image, display
analysis.plot_condition_accuracy_bars(df)
display(Image('experiments/adaptive_routing/fig_condition_accuracy.png'))


## 6 · WHERE each prompt helps or hurts — category × condition heatmap
The whole hypothesis in one figure: a prompt that is green on one row can be red on another.

In [ ]:
analysis.plot_category_heatmap(df)
display(Image('experiments/adaptive_routing/fig_category_heatmap.png'))
analysis.category_accuracy_matrix(df).round(2)


## 7 · The cost of reasoning — latency vs accuracy

In [ ]:
analysis.plot_latency_accuracy(df)
display(Image('experiments/adaptive_routing/fig_latency_accuracy.png'))


## 8 · The oracle the router chases — best fixed strategy per category
If the per-category best fixed strategy and the adaptive arm agree, the router is doing its job.

In [ ]:
analysis.best_strategy_per_category(df)


## 9 · Did the classifier label the reasoning shape correctly?
Routing accuracy + a confusion table (true category vs the category the classifier inferred).

In [ ]:
rep = analysis.routing_report(df)
print('overall routing accuracy:', round(rep['overall_routing_accuracy'], 3))
rep['confusion']


## 10 · Failure taxonomy — *why* each prompt got things wrong
Heuristic labels from the raw output: overthinking (recall Q over-reasoned), boundary_error (counting off-by-one), skipped_case (never enumerated), arithmetic_drift, no_answer_parsed, hallucinated/other.

In [ ]:
analysis.failure_taxonomy(df)


## 11 · Findings & recommendation

The full write-up lives in **`experiments/adaptive_routing_findings.md`** (publication-style section).

**Headline (to confirm on the real Colab run):**
- **Adaptive routing (D) should beat every single fixed prompt (A/B/C)** — because no one prompt is
  best across all categories, and D picks the per-category winner.
- **Structured enumeration is the cure for interval-counting / temporal questions** (the clock-chime
  family) but **hurts factual recall** (overthinking) — never make it the universal default.
- **Direct answering wins on factual_qa / commonsense**; **checklists win on logic / multi-hop**.
- The router's weakest spot is **multi_hop** (it surface-overlaps with factual recall).

**Recommendation for the live game:** in `notebooks/03_live_play.ipynb`, give the **Maths** pipeline a
per-question router that sends interval-counting / temporal / enumeration questions to
`structured_enumeration_cot` and leaves concept/stats questions on the current chain — exactly the
regime split `cot_v2`'s single brevity cap conflated.